# A/B Test

In [4]:
import pandas as pd
import numpy as np
import scipy.stats as st

In [5]:
likes = pd.read_csv('likes.csv')
views = pd.read_csv('views.csv')

In [6]:
likes

,user_id,post_id,timestamp
0,128381,4704,1654030804
1,146885,1399,1654030816
2,50948,2315,1654030828
3,14661,673,1654030831
4,37703,1588,1654030833
...,...,...,...
230171,31851,5964,1655243535
230172,51512,1498,1655243537
230173,34017,5009,1655243573
230174,13267,1787,1655243692


In [7]:
views

,user_id,exp_group,recommendations,timestamp
0,128381,control,[3644 4529 4704 5294 4808],1654030803
1,146885,test,[1399 1076 797 7015 5942],1654030811
2,50948,test,[2315 3037 1861 6567 4093],1654030825
3,37703,test,[2842 1949 162 1588 6794],1654030826
4,14661,test,[2395 5881 5648 3417 673],1654030829
...,...,...,...,...
193290,158267,test,[1733 6834 4380 1915 1627],1655240340
193291,63527,control,[2454 191 3873 6404 1588],1655240347
193292,52169,test,[1368 1709 1616 798 5305],1655240354
193293,142402,test,[5895 6984 1978 6548 6106],1655240373


In [9]:
duplic = views.groupby('user_id')['exp_group'].nunique()
duplic


,exp_group
user_id,
200,1
201,1
202,1
212,1
213,1
...,...
168538,1
168541,1
168544,1


In [11]:
more_then_2 = duplic[duplic > 1].index.tolist()
more_then_2

[25623, 55788, 142283, 148670]

In [13]:
if len(more_then_2) > 0:
  views_clear = views[~views['user_id'].isin(more_then_2)]
  likes_clear = likes[~likes['user_id'].isin(more_then_2)]

In [16]:
user_group = views_clear[['user_id', 'exp_group']].drop_duplicates()
user_group

,user_id,exp_group
0,128381,control
1,146885,test
2,50948,test
3,37703,test
4,14661,test
...,...,...
193267,80500,test
193270,149686,test
193285,3615,control
193288,119630,test


In [19]:
group_sizes = user_group['exp_group'].value_counts()
print(f'Group sizes: {group_sizes}')

n_control = group_sizes.get('control', 0)
n_test = group_sizes.get('test', 0)
n_total = n_control+n_test

print(f"Доля контрольной группы: {n_control / n_total:.4f}")

Group sizes: exp_group
test       32659
control    32350
Name: count, dtype: int64
Доля контрольной группы: 0.4976


In [21]:
test_result = st.binomtest(k=n_control, n=n_total, p = 0.5, alternative='two-sided')
print(f"P-value биномиального теста: {test_result.pvalue:.4f}")

if test_result.pvalue < 0.05:
    print("ВНИМАНИЕ: Разбиение статистически значимо отличается от 50/50 (система сплитования работает криво).")
else:
    print("УСПЕХ: Отклонения в размерах групп случайны, разбиение 50/50 работает корректно.")

P-value биномиального теста: 0.2271
УСПЕХ: Отклонения в размерах групп случайны, разбиение 50/50 работает корректно.


In [22]:
experiment_users = pd.DataFrame({'user_id': views_clear['user_id'].unique()})

likes_per_user = likes_clear.groupby('user_id').size().reset_index(name = 'likes_count')

user_metrics = experiment_users.merge(likes_per_user, on='user_id', how='left')
user_metrics['likes_count'] = user_metrics['likes_count'].fillna(0)

user_metrics.head(5)

,user_id,likes_count
0,128381,7.0
1,146885,4.0
2,50948,5.0
3,37703,6.0
4,14661,11.0


In [23]:
user_metrics['has_liked'] = (user_metrics['likes_count']>0).astype(int)
overall_conversion = user_metrics['has_liked'].mean()

print(f"\nДоля пользователей, сделавших хотя бы 1 лайк: {overall_conversion:.4f} "
      f"({overall_conversion * 100:.2f}%)")


Доля пользователей, сделавших хотя бы 1 лайк: 0.8948 (89.48%)


In [24]:
from statsmodels.stats.proportion import proportions_ztest

df_ab = user_metrics.merge(user_group, on='user_id', how='inner')
group_a = df_ab[df_ab['exp_group'] == 'control']
group_b = df_ab[df_ab['exp_group'] == 'test']

Тест 1: Z-критерий для долей (Конверсия)

In [25]:
successes = [group_a['has_liked'].sum(), group_b['has_liked'].sum()]
nobs =[len(group_a), len(group_b)]

stat_z, pval_z = proportions_ztest(successes, nobs)

print(f"Конверсия Control: {group_a['has_liked'].mean():.4f}")
print(f"Конверсия Test:    {group_b['has_liked'].mean():.4f}")
print(f"Z-test p-value:    {pval_z:.4f}")

Конверсия Control: 0.8913
Конверсия Test:    0.8982
Z-test p-value:    0.0045


Тест 2: Критерий Манна-Уитни (Число лайков)

In [27]:
stat_mw, pval_mw = st.mannwhitneyu(group_a['likes_count'], group_b['likes_count'], alternative='two-sided')

print(f"\nСреднее число лайков Control: {group_a['likes_count'].mean():.4f}")
print(f"Среднее число лайков Test:    {group_b['likes_count'].mean():.4f}")
print(f"Mann-Whitney p-value:         {pval_mw:.4f}")


Среднее число лайков Control: 3.4871
Среднее число лайков Test:    3.5926
Mann-Whitney p-value:         0.0000


## hitrate

In [28]:
import re

def parse_recs(val):
    if isinstance(val, str):
        return[int(x) for x in re.findall(r'\d+', val)]
    return val

views['recommendations'] = views['recommendations'].apply(parse_recs)
original_views_count = views.shape[0]

In [29]:
views_exploded = views.explode('recommendations').rename(columns={'recommendations': 'post_id'})

views_exploded = views_exploded.rename(columns={'timestamp': 'timestamp_view'})
likes = likes.rename(columns={'timestamp': 'timestamp_like'})


In [30]:
merged = views_exploded.merge(likes, on=['user_id', 'post_id'], how='left')

In [31]:
time_diff = merged['timestamp_like'] - merged['timestamp_view']

cond_valid = (time_diff >= 0) & (time_diff <= 3600)

merged['is_valid_like'] = cond_valid.astype(int)

In [32]:
hitrate_df = merged.groupby(['user_id', 'timestamp_view', 'exp_group'])['is_valid_like'].max().reset_index()

print(f"Исходно показов: {original_views_count}")
print(f"После обработки: {hitrate_df.shape[0]} (должно совпадать!)")

Исходно показов: 193295
После обработки: 193295 (должно совпадать!)


In [33]:
overall_hitrate = hitrate_df['is_valid_like'].mean()
print(f"\nОбщий HitRate: {overall_hitrate:.4f} ({overall_hitrate * 100:.2f}%)")

hitrate_by_group = hitrate_df.groupby('exp_group')['is_valid_like'].mean()
print("\nHitRate по группам:")
print(hitrate_by_group)


Общий HitRate: 0.7133 (71.33%)

HitRate по группам:
exp_group
control    0.706655
test       0.719843
Name: is_valid_like, dtype: float64


# Бакетный подход

In [35]:
import hashlib

def get_bucket(user_id, num_buckets=100):
    return int(hashlib.md5(str(user_id).encode()).hexdigest(), 16) % num_buckets

hitrate_df['bucket'] = hitrate_df['user_id'].apply(get_bucket)

bucket_metrics = hitrate_df.groupby(['exp_group', 'bucket']).agg(
    hits=('is_valid_like', 'sum'),
    shows=('is_valid_like', 'count')
).reset_index()

bucket_metrics['bucket_hitrate'] = bucket_metrics['hits'] / bucket_metrics['shows']

control_buckets = bucket_metrics[bucket_metrics['exp_group'] == 'control']['bucket_hitrate']
test_buckets = bucket_metrics[bucket_metrics['exp_group'] == 'test']['bucket_hitrate']


print(f"Средний HitRate по бакетам Control: {control_buckets.mean():.4f}")
print(f"Средний HitRate по бакетам Test:    {test_buckets.mean():.4f}")

stat_t, pval_t = st.ttest_ind(control_buckets, test_buckets, equal_var=False)

stat_mw, pval_mw = st.mannwhitneyu(control_buckets, test_buckets, alternative='two-sided')

print(f"\nT-test p-value:         {pval_t:.6f}")
print(f"Mann-Whitney p-value:   {pval_mw:.6f}")


Средний HitRate по бакетам Control: 0.7065
Средний HitRate по бакетам Test:    0.7199

T-test p-value:         0.000000
Mann-Whitney p-value:   0.000000
